# ODMR Spectroscopy: Batch Lorentzian Fitting and Physically Guided Pairing of Fully Resolved \(^{14}\mathrm{N}\) NV-Ensemble Resonances

This notebook processes **averaged CW-ODMR spectra** from a **bulk NV-center diamond ensemble** with **\(^{14}\mathrm{N}\)** hyperfine structure, for the three experimental scan folders **Variation Along X**, **Variation Along Y**, and **Variation Along Z**. For each averaged spectrum, it detects the **8 broad ODMR dips** expected from the four NV orientation classes in a magnetic field, fits all dips simultaneously with a **global Lorentzian triplet model**, extracts the **central hyperfine frequency** \(x_1\) of each dip together with the side components \(x_0\) and \(x_2\), and proposes **4 left-right dip pairs** using approximate symmetry about an inferred effective center frequency \(D_{\mathrm{eff}}\). The output is designed for the next reconstruction step, where changes in the fitted resonance frequencies under small lab-frame displacements are used to infer the magnetic response of the setup. This structure is physically motivated by the standard NV-ensemble ODMR picture in which a general field produces four resonance pairs, while each \(^{14}\mathrm{N}\) electronic transition is split into a hyperfine triplet. :contentReference[oaicite:0]{index=0}

This notebook is written specifically for the **fully resolved-spectrum regime**: it assumes that the ODMR trace contains **4 identifiable dips below** and **4 identifiable dips above** the zero-field splitting region near \(D \approx 2.87\ \text{GHz}\), so that each broad transition can be tracked separately. That assumption is appropriate for your measurement strategy, where the magnet configuration is chosen to split the resonances enough to resolve all eight broad dips. It is also exactly the regime in which extracting fitted dip centers and pairing the two transitions of each NV class is meaningful. By contrast, recent NV literature emphasizes that at low bias or in overlapping spectra, simple peak-based assignments can become ambiguous and one should then move to a fuller Hamiltonian-based forward model rather than rely on resolved-dip fitting alone. This notebook therefore **does not claim general validity outside the resolved regime**. :contentReference[oaicite:1]{index=1}

A **Lorentzian** line shape is used here on purpose. Recent work by **Dennis Lönard et al., “Limits of absolute vector magnetometry with NV centers in diamond”** recommends Voigt profiles when the goal is accurate linewidth metrology in the presence of both homogeneous and inhomogeneous broadening. However, the purpose of this notebook is different: it is a **batch extraction pipeline for resonance centers and pair assignments**, not a final linewidth study. For that task, a shared Lorentzian triplet model is a useful and deliberately simpler choice because it reduces fit degeneracy and makes automated center tracking across many files more stable. The shared \(^{14}\mathrm{N}\) hyperfine splitting is initialized near **\(2.16\ \text{MHz}\)**, consistent with the usual NV \(^{14}\mathrm{N}\) hyperfine scale, and the initial Lorentzian FWHM is set to **\(0.8\ \text{MHz}\)** as a realistic resolved-line starting value for CW-ODMR fitting. These are **initial guesses**, not imposed physical truths; the fitted values are determined from the data. :contentReference[oaicite:2]{index=2}

The pairing step is also chosen to match the physics of NV magnetometry. Instead of pairing dips only by sorted order, the notebook pairs one dip on the low-frequency side with one dip on the high-frequency side such that their midpoint is as consistent as possible with a common fitted \(D_{\mathrm{eff}}\). In other words, the code uses the fact that for a given NV orientation, the two ODMR transitions are approximately distributed as \(D_{\mathrm{eff}} \pm \Delta\), while allowing small deviations from perfect symmetry due to experimental imperfections, strain, temperature drift, or non-ideal field geometry. This makes the output more suitable for later reconstruction of the field response matrix from your lab-frame scans. :contentReference[oaicite:3]{index=3}

### Literature used in the design of this notebook
- **Pralekh Dubey et al.**, *“Magnetic field orientation dependence of continuous-wave optically detected magnetic resonance with nitrogen-vacancy ensembles”* — used here for the physical picture of four NV classes, eight broad ODMR transitions, and the warning that overlap/low-bias spectra can make naive assignments ambiguous. :contentReference[oaicite:4]{index=4}
- **Dennis Lönard et al.**, *“Limits of absolute vector magnetometry with NV centers in diamond”* — used here for the vector-magnetometry context and for the distinction between accurate linewidth modeling and practical extraction of resonance frequencies for field reconstruction. :contentReference[oaicite:5]{index=5}
- **Yuchun Zhu et al.**, *“Simulation of ODMR Spectra from Nitrogen-Vacancy Ensembles in Diamond for Electric Field Sensing”* — used here for the ensemble-NV spectral structure and the modeling picture in which each electronic transition can be treated as part of a hyperfine-resolved multiplet. :contentReference[oaicite:6]{index=6}

## Import Libraries


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from fitting_odmr.fitlorenzo_functions import (
    lorentzian, make_global_odmr_model, normalize_signal, smooth_signal,
    validate_frequency_units, detect_8_dip_center_guesses, fit_polynomial_baseline,
    pair_dips_by_symmetry, fit_global_odmr
)


---

## Part 1: Model Functions

### Physical model and literature-guided correction

We model the ODMR spectrum as a superposition of **8 hyperfine triplets**. Each triplet corresponds to one broadened electronic transition and consists of three equally spaced hyperfine components associated with the \(^{14}\mathrm{N}\) nuclear spin \(I=1\). This is consistent with current NV ODMR modeling papers that explicitly describe the \(3	imes 8\) structure for four NV classes with two electronic transitions per class.  
**Citations:** Zhu *et al.*, **“Simulation of ODMR Spectra from Nitrogen-Vacancy Ensembles in Diamond for Electric Field Sensing”** (arXiv:2301.04106); Dubey *et al.*, **“Magnetic field orientation dependence of continuous-wave optically detected magnetic resonance with nitrogen-vacancy ensembles”** (arXiv:2504.18478).



### Global ODMR model

\[
y(f)=1-\sum_{i=1}^{8} C_i\Big[w_{-1}\,\Phi(f;f_{c,i}-\Delta_{\mathrm{hf}})
+w_0\,\Phi(f;f_{c,i})
+w_{+1}\,\Phi(f;f_{c,i}+\Delta_{\mathrm{hf}})\Big],
\]

where \(\Phi\) is either a normalized **Voigt** or **Lorentzian** line shape, \(f_{c,i}\) is the center of the \(i\)-th broad dip, \(C_i\) is its contrast, and \(\Delta_{\mathrm{hf}}\) is a shared \(^{14}\mathrm{N}\) hyperfine splitting. Shared triplet weights are retained as a compact phenomenological model; a full Hamiltonian treatment would be the next step if resonances substantially overlap or mixing becomes important.  
**Citations:** Zhu *et al.*, **“Simulation of ODMR Spectra from Nitrogen-Vacancy Ensembles in Diamond for Electric Field Sensing”** (arXiv:2301.04106); Dubey *et al.*, **“Magnetic field orientation dependence of continuous-wave optically detected magnetic resonance with nitrogen-vacancy ensembles”** (arXiv:2504.18478); Lönard *et al.*, **“Limits of absolute vector magnetometry with NV centers in diamond”** (arXiv:2504.20750).



In [ ]:
# (All model and fitting functions are now imported from fitting_odmr/fitlorenzo_functions.py)
# See the import cell above for details.

---

## Part 2: Data Preprocessing Functions

### Normalization and smoothing

Before dip detection, the signal is normalized and lightly smoothed with a Savitzky–Golay filter. This does **not** define the fit model itself; it is only used to make initial dip detection more robust.

That correction is kept because smoothing for detection is standard practice, while the actual fit is still performed on the baseline-corrected spectrum rather than on the smoothed trace.  
**Literature note:** recent NV papers distinguish clearly between visual/detection processing and the actual physical forward model used for spectral interpretation.  
**Citations:** Dubey *et al.*, **“Magnetic field orientation dependence of continuous-wave optically detected magnetic resonance with nitrogen-vacancy ensembles”** (arXiv:2504.18478); Lönard *et al.*, **“Limits of absolute vector magnetometry with NV centers in diamond”** (arXiv:2504.20750).



In [ ]:
# (All normalization and smoothing functions are now imported from fitting_odmr/fitlorenzo_functions.py)
# See the import cell above for details.

In [ ]:
# (All frequency validation functions are now imported from fitting_odmr/fitlorenzo_functions.py)
# See the import cell above for details.

---

## Part 3: Broad ODMR Dip Detection

### Find 4 dips on each side of \(D\)

A practical assumption is that the measured spectrum contains **4 broad resonances below** and **4 broad resonances above** the zero-field splitting \(Dpprox 2.87\) GHz. That is physically appropriate when all four NV orientation classes are resolved and each contributes one \(m_s=0
ightarrow -1\) and one \(m_s=0
ightarrow +1\) transition.  
**Citations:** Dubey *et al.*, **“Magnetic field orientation dependence of continuous-wave optically detected magnetic resonance with nitrogen-vacancy ensembles”** (arXiv:2504.18478); Zhu *et al.*, **“Simulation of ODMR Spectra from Nitrogen-Vacancy Ensembles in Diamond for Electric Field Sensing”** (arXiv:2301.04106).



In [ ]:
# (All dip detection functions are now imported from fitting_odmr/fitlorenzo_functions.py)
# See the import cell above for details.

---

## Part 4: Baseline Fitting

A slowly varying optical background is removed before spectral fitting. Keeping a low-order polynomial baseline is a reasonable correction for CW-ODMR sweeps over a relatively narrow frequency window, where the fluorescence background changes slowly compared with the resonance features.

**Literature context:** recent ODMR papers typically compare the resonance model to a normalized or baseline-corrected fluorescence trace rather than to the raw detector voltage directly.  
**Citations:** Dubey *et al.*, **“Magnetic field orientation dependence of continuous-wave optically detected magnetic resonance with nitrogen-vacancy ensembles”** (arXiv:2504.18478); Lönard *et al.*, **“Limits of absolute vector magnetometry with NV centers in diamond”** (arXiv:2504.20750).



In [ ]:
# (All baseline fitting functions are now imported from fitting_odmr/fitlorenzo_functions.py)
# See the import cell above for details.

In [ ]:
# (All dip pairing functions are now imported from fitting_odmr/fitlorenzo_functions.py)
# See the import cell above for details.

---

## Part 5: Global ODMR Fitting

### Main workflow

The fitting function now:

1. crops the ODMR window,  
2. detects 8 broad dips,  
3. estimates a smooth baseline,  
4. builds a baseline-corrected spectrum,  
5. fits all 8 hyperfine triplets simultaneously with either a **Voigt** or **Lorentzian** model,  
6. returns parameter estimates **with uncertainties**,  
7. stores all intermediate arrays required for literature-style diagnostic plots.

### Why uncertainty extraction was added

Recent NV papers do not just report fitted center frequencies; they use those parameters downstream for field reconstruction, sensitivity estimates, and comparison across sweeps. That makes uncertainty propagation important, so the covariance from `curve_fit` is now unpacked explicitly.  
**Citations:** Lönard *et al.*, **“Limits of absolute vector magnetometry with NV centers in diamond”** (arXiv:2504.20750); Dubey *et al.*, **“Magnetic field orientation dependence of continuous-wave optically detected magnetic resonance with nitrogen-vacancy ensembles”** (arXiv:2504.18478).



In [ ]:
# (All global ODMR fitting functions are now imported from fitting_odmr/fitlorenzo_functions.py)
# See the import cell above for details.

In [ ]:
# Plot the baseline-corrected data and the fit
f_ghz = payload['f_ghz']
y_fit = payload['y_fit']
y_model = payload['y_model']
residuals = payload['residuals']

plt.figure(figsize=(10, 6))
plt.plot(f_ghz, y_fit, label='Baseline-corrected data (arb. units)')
plt.plot(f_ghz, y_model, label='Lorentzian global fit (arb. units)')
plt.xlabel('Frequency (GHz)')
plt.ylabel('Normalized ODMR (arb. units)')
plt.title('ODMR Fit (Frequency in GHz, Data in Arb. Units)')
plt.legend()
plt.show()

plt.figure(figsize=(10, 2))
plt.plot(f_ghz, residuals, label='Residuals (arb. units)')
plt.axhline(0, color='gray', linestyle='--')
plt.xlabel('Frequency (GHz)')
plt.ylabel('Residual (arb. units)')
plt.title('Fit Residuals (Frequency in GHz)')
plt.show()

In [25]:
def save_run_metadata(payload, results_df, pairs_df, out_json):
    """
    Save compact metadata needed for later reconstruction.
    """
    meta = {
        "D_nominal_GHz": float(payload["D_nominal_GHz"]),
        "D_eff_GHz": float(payload["D_eff_GHz"]),
        "delta_hf_MHz": float(payload["delta_hf_GHz"] * 1000.0),
        "delta_hf_err_MHz": float(payload["delta_hf_err_GHz"] * 1000.0),
        "lorentz_fwhm_MHz": float(payload["width_L_GHz"] * 1000.0),
        "lorentz_fwhm_err_MHz": float(payload["width_L_err_GHz"] * 1000.0),
        "weights_fit": {
            "w_m1": float(payload["weights_fit"][0]),
            "w_0": float(payload["weights_fit"][1]),
            "w_p1": float(payload["weights_fit"][2]),
        },
        "global_rms": float(payload["global_rms"]),
        "n_dips": int(len(results_df)),
        "n_pairs": int(len(pairs_df)),
    }

    with open(out_json, "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2)

In [26]:
# ------------------------------------------------------------
# Batch processing: fit all averaged files and save outputs
# ------------------------------------------------------------

def parse_offset_mm(name):
    """
    Extract signed displacement in mm from names like:
    '0cm', '0cm-0.1mm', '0cm+0.2mm', etc.
    """
    if name.strip() == "0cm":
        return 0.0

    m = re.search(r'([+-]?\d+(?:\.\d+)?)mm', name.replace(' ', ''))
    if m:
        return float(m.group(1))

    # fallback: if pattern is like 0cm-0.1mm
    m = re.search(r'0cm([+-]\d+(?:\.\d+)?)mm', name.replace(' ', ''))
    if m:
        return float(m.group(1))

    return np.nan


def summarize_run(base_name, variation, results_df, pairs_df, payload):
    """
    One summary row per run for later response-matrix reconstruction.
    """
    row = {
        "variation": variation,
        "run_name": base_name,
        "D_eff_GHz": payload["D_eff_GHz"],
        "delta_hf_MHz": payload["delta_hf_GHz"] * 1000.0,
        "lorentz_fwhm_MHz": payload["width_L_GHz"] * 1000.0,
        "global_rms": payload["global_rms"],
    }

    # Store central hyperfine positions x1 of the 8 fitted dips
    for i, v in enumerate(results_df["x1_GHz"].to_numpy(), start=1):
        row[f"dip_{i}_x1_GHz"] = v

    # Store proposed left/right pairs
    for _, r in pairs_df.iterrows():
        pid = int(r["pair_id"])
        row[f"pair_{pid}_left_x1_GHz"] = r["left_center_x1_GHz"]
        row[f"pair_{pid}_right_x1_GHz"] = r["right_center_x1_GHz"]
        row[f"pair_{pid}_splitting_MHz"] = r["splitting_MHz"]
        row[f"pair_{pid}_center_minus_D_eff_MHz"] = r["pair_center_minus_D_eff_MHz"]

    return row


variations = ["Variation Along X", "Variation Along Y", "Variation Along Z"]
root_in = Path("../avareged_data/outputs")
root_out = Path("./batch_fit_outputs_lorentzian")

root_out.mkdir(parents=True, exist_ok=True)

all_results = {}
summary_rows = []

for variation in variations:
    data_dir = root_in / variation
    if not data_dir.exists():
        print(f"Skipping missing folder: {data_dir}")
        continue

    out_dir = root_out / variation
    out_dir.mkdir(parents=True, exist_ok=True)

    files = sorted([f for f in data_dir.iterdir() if f.name.endswith("_averaged.csv")])

    stats_path = data_dir / "averaging_statistics.csv"
    stats_df = pd.read_csv(stats_path) if stats_path.exists() else None

    for file_path in files:
        base_name = file_path.name.replace("_averaged.csv", "")
        offset_mm = parse_offset_mm(base_name)

        print(f"Processing {variation} / {file_path.name}")

        df = pd.read_csv(file_path, sep="\t")

        sigma = None
        if stats_df is not None and "Displacement" in stats_df.columns and "Std Dev" in stats_df.columns:
            match = stats_df.loc[stats_df["Displacement"] == base_name, "Std Dev"]
            if len(match) > 0 and np.isfinite(match.values[0]):
                sigma = np.full(len(df), float(match.values[0]))

        try:
            results_df, pairs_df, payload = fit_global_odmr(
                df,
                baseline_deg=2,
                baseline_mask_halfwidth_GHz=0.006,
                prominence=0.01,
                sigma=sigma,
            )

            run_out = out_dir / base_name
            run_out.mkdir(parents=True, exist_ok=True)

            # main fit tables
            results_df.to_csv(run_out / "fitted_dips.csv", index=False)
            pairs_df.to_csv(run_out / "paired_dips.csv", index=False)

            # dense curve for downstream inspection
            curve_df = pd.DataFrame({
                "frequency_Hz": payload["f_hz"],
                "frequency_GHz": payload["f_ghz"],
                "y_raw": payload["y_raw"],
                "y_norm": payload["y_norm"],
                "y_smooth": payload["y_smooth"],
                "baseline_raw": payload["baseline_raw"],
                "y_fit": payload["y_fit"],
                "y_model": payload["y_model"],
                "residual": payload["residuals"],
            })
            curve_df.to_csv(run_out / "fit_curve_and_residuals.csv", index=False)

            # metadata
            save_run_metadata(payload, results_df, pairs_df, run_out / "fit_metadata.json")

            # saved plot only
            save_fit_diagnostic_plot(
                payload,
                out_png=run_out / "fit_and_residuals.png",
                title=f"{variation} | {base_name}"
            )

            all_results[(variation, base_name)] = {
                "results_df": results_df,
                "pairs_df": pairs_df,
                "payload": payload,
                "offset_mm": offset_mm,
            }

            row = summarize_run(base_name, variation, results_df, pairs_df, payload)
            row["offset_mm"] = offset_mm
            summary_rows.append(row)

        except Exception as exc:
            fail_dir = out_dir / base_name
            fail_dir.mkdir(parents=True, exist_ok=True)
            with open(fail_dir / "fit_failed.txt", "w", encoding="utf-8") as f:
                f.write(str(exc))
            print(f"FAILED: {variation} / {base_name}: {exc}")

summary_df = pd.DataFrame(summary_rows).sort_values(["variation", "offset_mm"])
summary_df.to_csv(root_out / "summary_all_runs.csv", index=False)
display(summary_df.head())

Processing Variation Along X / 0cm-0.0mm_averaged.csv
Processing Variation Along X / 0cm-0.1mm_averaged.csv
Processing Variation Along X / 0cm-0.2mm_averaged.csv
Processing Variation Along X / 0cm-0.3mm_averaged.csv
Processing Variation Along X / 0cm-0.4mm_averaged.csv
Processing Variation Along Y / 0cm-0.0mm_averaged.csv
Processing Variation Along Y / 0cm-0.1mm_averaged.csv
Processing Variation Along Y / 0cm-0.2mm_averaged.csv
Processing Variation Along Y / 0cm-0.3mm_averaged.csv
Processing Variation Along Y / 0cm-0.4mm_averaged.csv
Processing Variation Along Z / 6.8cm-0.0mm_averaged.csv
Processing Variation Along Z / 6.8cm-0.1mm_averaged.csv
Processing Variation Along Z / 6.8cm-0.2mm_averaged.csv
Processing Variation Along Z / 6.8cm-0.3mm_averaged.csv
Processing Variation Along Z / 6.8cm-0.4mm_averaged.csv
Processing Variation Along Z / 6.8cm-0.5mm_averaged.csv


,variation,run_name,D_eff_GHz,delta_hf_MHz,lorentz_fwhm_MHz,global_rms,dip_1_x1_GHz,dip_2_x1_GHz,dip_3_x1_GHz,dip_4_x1_GHz,dip_5_x1_GHz,dip_6_x1_GHz,dip_7_x1_GHz,dip_8_x1_GHz,pair_1_left_x1_GHz,pair_1_right_x1_GHz,pair_1_splitting_MHz,pair_1_center_minus_D_eff_MHz,pair_2_left_x1_GHz,pair_2_right_x1_GHz,pair_2_splitting_MHz,pair_2_center_minus_D_eff_MHz,pair_3_left_x1_GHz,pair_3_right_x1_GHz,pair_3_splitting_MHz,pair_3_center_minus_D_eff_MHz,pair_4_left_x1_GHz,pair_4_right_x1_GHz,pair_4_splitting_MHz,pair_4_center_minus_D_eff_MHz,offset_mm
4,Variation Along X,0cm-0.4mm,2.871576,2.199619,1.533325,0.000643,2.807770,2.818928,2.835220,2.846094,2.898628,2.908747,2.923713,2.933508,2.807770,2.933508,125.738095,-0.936995,2.818928,2.923713,104.785094,-0.255599,2.835220,2.908747,73.526805,0.407387,2.846094,2.898628,52.534524,0.785207,-0.4
3,Variation Along X,0cm-0.3mm,2.871544,2.237182,1.579573,0.000729,2.808500,2.819427,2.835074,2.845667,2.899066,2.908740,2.923169,2.932712,2.808500,2.932712,124.211408,-0.938357,2.819427,2.923169,103.741528,-0.246403,2.835074,2.908740,73.666087,0.362643,2.845667,2.899066,53.399657,0.822117,-0.3
2,Variation Along X,0cm-0.2mm,2.871539,2.188336,1.482833,0.000719,2.809269,2.819939,2.834804,2.845249,2.899307,2.909024,2.922639,2.932084,2.809269,2.932084,122.814939,-0.863177,2.819939,2.922639,102.700289,-0.250149,2.834804,2.909024,74.220529,0.374400,2.845249,2.899307,54.058116,0.738926,-0.2
1,Variation Along X,0cm-0.1mm,2.871489,2.192951,1.504535,0.000550,2.809967,2.820412,2.834685,2.844775,2.899702,2.908968,2.922067,2.931334,2.809967,2.931334,121.366230,-0.838336,2.820412,2.922067,101.655332,-0.249408,2.834685,2.908968,74.283802,0.337753,2.844775,2.899702,54.927207,0.749991,-0.1
0,Variation Along X,0cm-0.0mm,2.871453,2.218733,1.546208,0.000725,2.810709,2.820885,2.834467,2.844376,2.899918,2.909126,2.921594,2.930545,2.810709,2.930545,119.836066,-0.825603,2.820885,2.921594,100.708983,-0.212674,2.834467,2.909126,74.658799,0.343731,2.844376,2.899918,55.541525,0.694546,-0.0
